# 토크나이저 학습 + 검증 노트북 (로컬)

SentencePiece **Unigram / NFKC / vocab 32768** 토크나이저를 셀 단위로 실행하며 확인한다.
(vocab 을 10,240 → 32,768 로 올린 근거는 [docs/model_config_review.md](docs/model_config_review.md) §4 — 10k 는 한글 1음절 piece 1,645개뿐이라 byte-fallback 2.3%, 한글 토큰의 45% 가 음절 단위로 쪼개졌다.)

- 커널: 이 리포의 `.venv` 를 선택할 것 (`sentencepiece`, `pandas` 필요)
- 입력: `train.jsonl` (없으면 `train.tar.xz` 에서 자동 해제)
- 산출물: `tokenizer/spm.model` — 이후 모델 학습이 그대로 사용하며 git 에 커밋한다. `model.py` 의 `vocab_size=32768` 과 일치해야 하고, 데이터 캐시는 파일명에 vocab 이 들어가 자동 재생성된다
- 이미 32k 버전 `tokenizer/spm.model` 이 있으면 1~2번 셀을 건너뛰고 3번부터 실행해도 된다

In [1]:
# 0) 준비
from pathlib import Path
import json, unicodedata
import pandas as pd

from data import ensure_dataset
from tokenizer.train_tokenizer import extract_corpus, train, SPECIAL_TURN_TOKENS

ROOT = Path.cwd()
CORPUS = ROOT / 'cache' / 'tokenizer_corpus.txt'
CORPUS.parent.mkdir(exist_ok=True)
MODEL_PATH = ROOT / 'tokenizer' / 'spm.model'

jsonl = ensure_dataset(ROOT, 'train')   # 없으면 train.tar.xz 자동 해제
print('입력:', jsonl, f'({jsonl.stat().st_size / 1e6:.0f} MB)')

입력: /home/pc/project/korean_sllm/train.jsonl (406 MB)


In [2]:
# 1) 코퍼스 추출 - user/assistant 텍스트를 한 줄씩 뽑는다 (I/O 위주, 수십 초)
%time n = extract_corpus(jsonl, CORPUS)
print(f'코퍼스 {n:,}줄, {CORPUS.stat().st_size / 1e6:.0f} MB')

with CORPUS.open(encoding='utf-8') as f:
    for _, line in zip(range(5), f):
        print(' |', line.strip()[:80])

CPU times: user 2.45 s, sys: 2.01 s, total: 4.46 s
Wall time: 4.6 s
코퍼스 603,338줄, 394 MB
 | 양파는 어떤 식물 부위인가요? 그리고 고구마는 뿌리인가요?
 | 양파는 잎이 아닌 식물의 줄기 부분입니다. 고구마는 식물의 뿌리 부분입니다.   식물의 부위의 구분에 대해 궁금해하는 분이라면 분명 이 질문에 
 | 스웨터의 유래는 어디에서 시작되었나요?
 | 스웨터의 유래는 14세기경 북유럽항구지역에서 어망을 짜던 기술을 의복에 활용하면서 시작되었습니다. 노동자들의 방한복에서 시작된 스웨터는 여가생활
 | 토성의 고리가 빛의 띠로 보이는 이유는 무엇인가요?    토성의 고리는 얼음과 같은 여러 물질로 이루어져 있다고 알고 있는데, 카시니가 찍은 사


In [3]:
import os
# 2) SentencePiece Unigram 학습 (수 분 소요, 로그가 아래에 출력된다)
#    num_threads 기본값 = os.cpu_count() -> EM 학습 단계가 모든 코어를 사용한다
%time model_path = train(CORPUS, vocab_size=32768, num_threads=os.cpu_count())
print('저장:', model_path)

I0000 00:00:1788192494.194424  110952 sentencepiece_trainer.cc:105] Starts training with : 
trainer_spec {
  input: /home/pc/project/korean_sllm/cache/tokenizer_corpus.txt
  input_format: 
  model_prefix: /home/pc/project/korean_sllm/tokenizer/spm
  model_type: UNIGRAM
  vocab_size: 32768
  self_test_sample_size: 0
  character_coverage: 0.9999
  input_sentence_size: 2000000
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 32768
  num_threads: 24
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 1
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 1
  user_defined_symbols: <start_of_turn>
  user_defined_symbols: <end_of_turn>
  required_chars: 
  byte_fallback: 1
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_al

CPU times: user 1h 25min 39s, sys: 13.9 s, total: 1h 25min 53s
Wall time: 17min 27s
저장: /home/pc/project/korean_sllm/tokenizer/spm.model


In [4]:
# 3) 로드 + 기본 정보
import sentencepiece as spm
sp = spm.SentencePieceProcessor(model_file=str(MODEL_PATH))

print('vocab size :', sp.get_piece_size())
print('pad/bos/eos/unk id:', sp.pad_id(), sp.bos_id(), sp.eos_id(), sp.unk_id())
for tok in SPECIAL_TURN_TOKENS:
    print(f'{tok:>16} -> id {sp.piece_to_id(tok)}')
print('\n앞 20개 piece:', [sp.id_to_piece(i) for i in range(20)])

vocab size : 32768
pad/bos/eos/unk id: 0 1 2 3
 <start_of_turn> -> id 4
   <end_of_turn> -> id 5

앞 20개 piece: ['<pad>', '<bos>', '<eos>', '<unk>', '<start_of_turn>', '<end_of_turn>', '<0x00>', '<0x01>', '<0x02>', '<0x03>', '<0x04>', '<0x05>', '<0x06>', '<0x07>', '<0x08>', '<0x09>', '<0x0A>', '<0x0B>', '<0x0C>', '<0x0D>']


In [5]:
# 4) 인코딩/디코딩 round-trip 확인
samples = [
    '안녕하세요, 국민건강보험법 제5조를 요약해 주세요.',
    '혈압이 140/90 mmHg 이상이면 고혈압입니다.',
    'The quick brown fox jumps over 13 lazy dogs.',
    '이모지 😊 와 한자 漢字, 특수문자 ㈜·※ 도 깨지지 않아야 한다.',
]
for text in samples:
    ids = sp.encode(text)
    decoded = sp.decode(ids)
    ok = decoded == unicodedata.normalize('NFKC', text)   # NFKC 정규화 후 일치해야 정상
    print(f"[{'OK ' if ok else 'DIFF'}] {len(ids):3d} tokens | {text}")
    print('      pieces:', sp.encode(text, out_type=str))
    if not ok:
        print('      decoded:', decoded)

[OK ]  10 tokens | 안녕하세요, 국민건강보험법 제5조를 요약해 주세요.
      pieces: ['▁안녕하세요', ',', '▁국민건강보험법', '▁제', '5', '조', '를', '▁요약해', '▁주세요', '.']
[OK ]  13 tokens | 혈압이 140/90 mmHg 이상이면 고혈압입니다.
      pieces: ['▁혈압이', '▁', '1', '4', '0', '/', '9', '0', '▁mmHg', '▁이상이면', '▁고혈압', '입니다', '.']
[OK ]  21 tokens | The quick brown fox jumps over 13 lazy dogs.
      pieces: ['▁The', '▁', 'quick', '▁', 'br', 'own', '▁f', 'ox', '▁j', 'ump', 's', '▁over', '▁', '1', '3', '▁', 'lazy', '▁', 'dog', 's', '.']
[OK ]  26 tokens | 이모지 😊 와 한자 漢字, 특수문자 ㈜·※ 도 깨지지 않아야 한다.
      pieces: ['▁이', '모', '지', '▁', '😊', '▁', '와', '▁한자', '▁', '漢', '字', ',', '▁특수', '문자', '▁(', '주', ')', '·', '※', '▁', '도', '▁깨', '지지', '▁않아야', '▁한다', '.']


In [6]:
# 5) 챗 템플릿 인코딩 확인 - 학습 시 실제로 쓰는 형태 (mask=1 구간만 손실 계산)
from data import encode_sample

ids, mask = encode_sample(sp, '감기에 걸렸을 때 어떻게 해야 하나요?', '충분한 휴식과 수분 섭취가 중요합니다.')
df = pd.DataFrame({'id': ids, 'piece': [sp.id_to_piece(i) for i in ids], 'loss_mask': mask})
print(f'총 {len(ids)} tokens, 손실 계산 대상 {sum(mask)} tokens')
df.T

총 27 tokens, 손실 계산 대상 8 tokens


,0,1,2,3,4,5,6,7,8,9,...,17,18,19,20,21,22,23,24,25,26
id,1,262,4,1500,16,1259,3631,17960,276,351,...,1578,16,1380,10915,3017,17490,495,263,5,2
piece,<bos>,▁,<start_of_turn>,user,<0x0A>,감,기에,▁걸렸,을,▁때,...,model,<0x0A>,▁충분한,▁휴식과,▁수분,▁섭취가,▁중요합니다,.,<end_of_turn>,<eos>
loss_mask,0,0,0,0,0,0,0,0,0,0,...,0,0,1,1,1,1,1,1,1,1


In [7]:
# 6) 실데이터 토큰 통계 - 파일 전체에서 등간격 표본 4,000개
#    주의: train.jsonl 은 소스 파일명 순으로 이어 붙인 것(셔플 안 됨)이라 "앞 N줄"만 보면
#    KoAlpaca 한 소스만 보게 된다. 전체 분포(전수 집계)는 docs/seq_len_review.md 참조.
N_SAMPLE = 4000
n_lines = sum(1 for _ in jsonl.open(encoding='utf-8'))
stride = max(n_lines // N_SAMPLE, 1)

rows = []
with jsonl.open(encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i % stride:
            continue
        try:
            obj = json.loads(line)
        except json.JSONDecodeError:
            continue
        ids, mask = encode_sample(sp, obj.get('user', ''), obj.get('assistant', ''))
        rows.append({'tokens': len(ids), 'prompt': len(ids) - sum(mask), 'answer': sum(mask),
                     'chars': len(obj.get('user', '')) + len(obj.get('assistant', ''))})
stats = pd.DataFrame(rows)
stats['chars_per_token'] = stats['chars'] / stats['tokens']
print(f'{n_lines:,}줄 중 {len(stats):,}개 표본 (stride {stride})')
print(stats.describe(percentiles=[.5, .9, .95, .99]).round(1))

for L in (512, 1024, 2048):
    print(f'seq_len {L:5d} 이내 샘플: {(stats.tokens <= L).mean() * 100:5.1f}%')
print(f"\n=> 학습 seq_len 2048: p99 {stats.tokens.quantile(.99):.0f} tokens, 초과 {(stats.tokens > 2048).mean() * 100:.2f}%"
      f" | 답변 p90 {stats.answer.quantile(.9):.0f} tokens (추론 max_new_tokens 512 이상 권장)"
      f" | 평균 압축률 {stats['chars_per_token'].mean():.2f} chars/token")

301,669줄 중 4,023개 표본 (stride 75)
       tokens  prompt  answer   chars  chars_per_token
count  4023.0  4023.0  4023.0  4023.0           4023.0
mean    250.5    55.4   195.1   615.4              2.3
std     236.0   145.8   167.4   614.5              0.4
min      14.0    11.0     3.0     3.0              0.2
50%     170.0    23.0   133.0   382.0              2.3
90%     471.0    96.0   385.8  1210.4              2.9
95%     661.0   143.9   536.0  1715.0              3.0
99%    1144.4   823.3   819.8  3099.2              3.3
max    3439.0  3280.0  1750.0  6257.0              4.9
seq_len   512 이내 샘플:  91.2%
seq_len  1024 이내 샘플:  98.5%
seq_len  2048 이내 샘플:  99.9%

=> 학습 seq_len 2048: p99 1144 tokens, 초과 0.15% | 답변 p90 386 tokens (추론 max_new_tokens 512 이상 권장) | 평균 압축률 2.33 chars/token


In [8]:
# 7) vocab 살펴보기 - 어떤 조각들이 학습됐는지
pieces = [sp.id_to_piece(i) for i in range(sp.get_piece_size())]

longest = sorted(pieces, key=len, reverse=True)[:20]
print('가장 긴 piece 20개:')
for piece in longest:
    print('  ', piece)

n_byte = sum(piece.startswith('<0x') for piece in pieces)
print(f'\nbyte-fallback piece: {n_byte}개 (256개면 정상)')
print('한국어 piece 예시:', [p for p in pieces[100:3000] if any("가" <= c <= "힣" for c in p)][:30])

가장 긴 piece 20개:
   ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
   ----------------
   \_\_\_\_\_\_\_\_
   \*\*\*\*\*\*\*\*
   addEventListener
   ---------------+
   AzLocalRightName
   ordNetLemmatizer
   ---------------
   ▁TfidfVectorizer
   ^^^^^^^^^^^^^^^^
   rgentinaVsFrance
   querySelectorAll
   connectionString
   <start_of_turn>
   ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
   springframework
   ▁ganztagsschule

byte-fallback piece: 256개 (256개면 정상)
한국어 piece 예시: ['의', '▁수', '이', '에', '을', '는', '▁있습니다', '를', '가', '은', '과', '에서', '와', '로', '▁이', '▁및', '▁있는', '으로', '▁대한', '도', '입니다', '한', '습니다', '▁위해', '▁데', '년', '▁더', '▁합니다', '▁또는', '인']
